# Trabalho 1, Tema 2: Vento favorável → ressurgência costeira (Cabo Frio)

**ACA0430 — Meteorologia Sinótica e Aplicações à Oceanografia (IAG/USP), 2º semestre de 2026**

**Objetivo.** Montar as figuras de base de um estudo de caso de **ressurgência costeira** em Cabo Frio (RJ): a situação sinótica que mantém o vento de NE, a resposta da temperatura e da salinidade da superfície, a estrutura vertical da água fria na plataforma e a série temporal de vento, transporte de Ekman e TSM. O exemplo padrão é **janeiro de 2016** (08 a 26/01).

**Por que este evento.** Há dados *in situ* publicados: Ramos e Silva et al. (2023) fizeram um cruzeiro em **20–21/01/2016** e encontraram **ACAS junto ao fundo** (13,9–14,7 °C). Mesmo assim, classificaram a ressurgência como **fraca**, porque o vento de NE **enfraqueceu dias antes**, durante o El Niño 2015/16. É um bom caso para comparar reanálise e observação.

**Ligação com as aulas.**

- A borda oeste do **Anticiclone Subtropical do Atlântico Sul (ASAS)** mantém **vento de NE** paralelo à costa.
- No Hemisfério Sul, o **transporte de Ekman é para a esquerda do vento**: vento de NE em Cabo Frio leva a água de superfície **para fora da costa**.
- Para repor essa água, sobe a **ACAS** (Água Central do Atlântico Sul), **fria e rica em nutrientes**.
- O **rotacional do vento** (bombeamento de Ekman) também contribui (Castelao & Barth, 2006).
- A **mudança de orientação da costa** em Cabo Frio, de N–S para E–W, é essencial: a oeste do cabo, a costa fica quase alinhada com o vento de NE.
- A passagem de **frentes frias** (vento de S/SW) **interrompe** a ressurgência.

**Este notebook é um ponto de partida**, não uma receita fechada.

**Dados:** ERA5 (atmosfera, horário, 0,25°) e GLORYS12 (oceano, diário, 1/12°), lidos de armazenamentos **Zarr na nuvem** (ARCO).

## 1. Instalação dos pacotes

O Colab já traz `numpy`, `pandas`, `xarray` e `matplotlib`. Os pacotes abaixo **não vêm pré-instalados**:

- `copernicusmarine`: *toolbox* oficial do Copernicus Marine (GLORYS12 e WAVERYS);
- `gcsfs` e `zarr`: leitura do ERA5 direto do Google Cloud Storage, em formato Zarr;
- `cartopy`: mapas com linha de costa e projeções;
- `cmocean`: paletas de cores pensadas para oceanografia.

A instalação leva cerca de 1 minuto. Em geral **não é preciso reiniciar** a sessão. Se aparecer erro de importação na célula seguinte, use *Ambiente de execução → Reiniciar sessão* e rode o notebook de novo a partir daqui.

Fora do Colab (no seu computador), a instalação automática é pulada: instale os pacotes no seu ambiente antes.

In [ ]:
import sys

# Detecta se estamos no Google Colab
EM_COLAB = "google.colab" in sys.modules

# Pacotes que não vêm pré-instalados no Colab
if EM_COLAB:
    !pip install -q copernicusmarine gcsfs zarr cartopy cmocean

## 2. Importações

Importamos tudo o que vamos usar e imprimimos as versões dos pacotes principais. **Se algo der errado, copie esta saída ao pedir ajuda**: muitos problemas vêm de diferenças de versão.

In [ ]:
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import zarr
import gcsfs
import copernicusmarine

warnings.filterwarnings("ignore", category=FutureWarning)

for nome, modulo in [("xarray", xr), ("zarr", zarr), ("gcsfs", gcsfs),
                     ("copernicusmarine", copernicusmarine), ("cartopy", cartopy)]:
    print(f"{nome:18s} {modulo.__version__}")

## 3. Credenciais do Copernicus Marine (CMEMS)

O **ERA5** que usamos aqui é **público**: não precisa de conta. Já o **CMEMS** (GLORYS12 e WAVERYS) exige uma **conta gratuita**.

**Nunca escreva sua senha no notebook.** Se você compartilhar o notebook, a senha vai junto.

### Passo a passo (uma vez só)

1. **Crie a conta** em https://data.marine.copernicus.eu/register. Preencha nome, e-mail, usuário e senha e confirme pelo link que chega no e-mail. Guarde o **nome de usuário**, que não é o e-mail.
2. **Crie os segredos no Colab:**
   - na barra lateral esquerda do Colab, clique no ícone de **chave** ("Secrets" / "Segredos");
   - clique em **"+ Adicionar novo segredo"**;
   - em *Nome*, escreva `COPERNICUSMARINE_SERVICE_USERNAME` e, em *Valor*, seu usuário do Copernicus Marine;
   - adicione um segundo segredo, `COPERNICUSMARINE_SERVICE_PASSWORD`, com a sua senha;
   - ative a chave **"Acesso ao notebook"** nos dois segredos.
3. Rode a célula abaixo. Se o Colab perguntar se o notebook pode acessar os segredos, clique em **"Conceder acesso"**.

Se você não criar os segredos, a célula pede usuário e senha numa caixa de texto (`getpass`), que não fica gravada no notebook.

Os segredos ficam na sua conta Google e servem para todos os seus notebooks do Colab.

In [ ]:
usuario = os.environ.get("COPERNICUSMARINE_SERVICE_USERNAME")
senha = os.environ.get("COPERNICUSMARINE_SERVICE_PASSWORD")

# 1) Segredos do Colab (ícone de chave na barra lateral)
if EM_COLAB and not usuario:
    from google.colab import userdata
    try:
        usuario = userdata.get("COPERNICUSMARINE_SERVICE_USERNAME")
        senha = userdata.get("COPERNICUSMARINE_SERVICE_PASSWORD")
    except Exception:  # segredo não criado ou acesso não concedido
        usuario, senha = None, None

if not usuario and copernicusmarine.login(check_credentials_valid=True):
    # Fora do Colab: login já salvo neste computador (comando `copernicusmarine login`)
    print("Usando o login do Copernicus Marine já salvo neste computador.")
else:
    # 2) Alternativa: perguntar usuário e senha (a senha não aparece na tela)
    if not usuario:
        from getpass import getpass
        usuario = input("Usuário do Copernicus Marine: ")
        senha = getpass("Senha do Copernicus Marine: ")

    # O toolbox também lê estas variáveis de ambiente
    os.environ["COPERNICUSMARINE_SERVICE_USERNAME"] = usuario
    os.environ["COPERNICUSMARINE_SERVICE_PASSWORD"] = senha

    # 3) Login (uma vez por sessão)
    ok = copernicusmarine.login(username=usuario, password=senha, force_overwrite=True)
    print("Login OK" if ok else "Falha no login: confira usuário e senha")

del senha  # não deixa a senha numa variável visível

## 4. Parâmetros do estudo de caso

**Esta é a única célula que você precisa alterar.**

- `REGIAO`: domínio "local" para o oceano, a tensão do vento e o bombeamento de Ekman (25°S–21°S, 45°W–40°W).
- `REGIAO_SINOTICA`: domínio maior, só para os mapas sinóticos (para ver a borda oeste do ASAS e as frentes).
- `INICIO`, `FIM`: janela do evento (UTC).
- `PASSO_MAPAS_H`: intervalo entre os mapas sinóticos, em horas. O dia do pico (12 UTC) é sempre incluído.
- `SERIE_INICIO`, `SERIE_FIM`: período da série temporal no ponto (pode ser mais longo que o evento).
- `PONTO`: local da série temporal e origem do transecto vertical (plataforma interna, a oeste de Arraial do Cabo).
- `ANGULO_COSTA_GRAUS`: orientação da costa (convenção abaixo).
- `DIA_ANTES`, `DIA_PICO`: dias usados para comparar "antes" e "no pico". `DIA_PICO = None` escolhe automaticamente o dia de **TSM mínima** no ponto (GLORYS12).
- Transecto: distância inicial e final (km) a partir do ponto, na direção perpendicular à costa, e profundidade máxima.

**Convenção do ângulo da costa.** `ANGULO_COSTA_GRAUS` é o azimute (graus a partir do norte, sentido horário) de uma direção **ao longo da costa com o oceano à DIREITA**. A oeste de Cabo Frio (Arraial do Cabo–Saquarema), a costa corre de oeste para leste com o oceano ao sul: cerca de 90°. A direção **para fora da costa** é α + 90° (cerca de 180°, para o sul).

```
          N
          ↑
   terra  |   costa (α ≈ 90°)
  ────────●──────────→ L
          |   oceano (à direita)
          ↓ para fora da costa (α + 90°)
```

**Verifique no mapa** se o ângulo representa bem a costa perto do seu ponto, porque ele muda muito em volta do cabo.

In [ ]:
# >>> ALUNO: domínio local (oceano, tensão do vento, bombeamento de Ekman)
REGIAO = dict(lon_o=-45.0, lon_l=-40.0, lat_s=-25.0, lat_n=-21.0)

# >>> ALUNO: domínio dos mapas sinóticos (maior, para ver o ASAS e as frentes)
REGIAO_SINOTICA = dict(lon_o=-60.0, lon_l=-25.0, lat_s=-40.0, lat_n=-10.0)

# >>> ALUNO: janela do evento (UTC)
INICIO = "2016-01-08T00:00"
FIM = "2016-01-26T21:00"

# >>> ALUNO: intervalo entre os mapas sinóticos (horas)
PASSO_MAPAS_H = 48

# >>> ALUNO: período da série temporal no ponto
SERIE_INICIO = "2016-01-03T00:00"
SERIE_FIM = "2016-01-31T21:00"

# >>> ALUNO: ponto na plataforma ao largo de Cabo Frio e orientação da costa
PONTO = dict(lon=-42.40, lat=-23.05, nome="Plataforma a oeste de Arraial do Cabo")
ANGULO_COSTA_GRAUS = 90.0

# >>> ALUNO: dias para comparar "antes" e "no pico" (None = dia de TSM mínima no ponto)
DIA_ANTES = "2016-01-08"
DIA_PICO = None

# >>> ALUNO: transecto perpendicular à costa (km a partir do ponto; negativo = em direção à costa)
TRANSECTO_INICIO_KM = -5.0
TRANSECTO_FIM_KM = 100.0
PROF_MAX = 200.0        # profundidade máxima da seção (m)

# Constantes físicas
RHO_AGUA = 1025.0       # densidade da água do mar (kg m-3)
OMEGA = 7.2921e-5       # velocidade angular da Terra (rad s-1)
RAIO_TERRA = 6.371e6    # m

### Pastas de saída

No Colab, os recortes de dados (NetCDF) vão para `/content/dados/` e as figuras para `/content/figuras/`. **Atenção: a pasta `/content` é apagada quando a sessão termina.** No fim do notebook há uma célula opcional para copiar tudo para o seu Google Drive.

In [ ]:
PASTA_BASE = Path("/content") if EM_COLAB else Path.cwd() / "saida"
PASTA_DADOS = PASTA_BASE / "dados"
PASTA_FIGURAS = PASTA_BASE / "figuras"
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
print("Dados:  ", PASTA_DADOS)
print("Figuras:", PASTA_FIGURAS)

## 5. Funções auxiliares

Poucas funções, usadas várias vezes ao longo do notebook:

- `lon_para_360` / `lon_para_180`: o ERA5 usa longitude de **0 a 360°** (o Brasil fica entre ~305° e ~330°). Recortamos com a longitude convertida e só voltamos para −180..180 **depois** do recorte, na hora de plotar;
- `checar_tamanho`: mostra quantos MB o recorte vai ocupar **antes** de carregar;
- `mapa_base`: costa, fronteiras, grade e (opcionalmente) continente em cinza;
- `passo_vetores`: reduz o número de setas para o mapa não virar um borrão;
- `ponto_oceano_mais_proximo`: perto da costa, a grade dos modelos oceânicos tem pontos de terra (NaN). Esta função escolhe o ponto **de oceano** mais próximo;
- `rodape` e `salvar_figura`: fonte dos dados e exportação em PNG.

In [ ]:
def lon_para_360(lon):
    """Converte longitude de -180..180 para 0..360 (convenção do ERA5)."""
    return lon % 360


def lon_para_180(ds, nome="longitude"):
    """Converte a coordenada de longitude de 0..360 para -180..180 e reordena (para plotar)."""
    ds = ds.assign_coords({nome: ((ds[nome] + 180) % 360) - 180})
    return ds.sortby(nome)


def checar_tamanho(ds, nome, limite_mb=1500):
    """Imprime o tamanho estimado do recorte ANTES de carregar na memória."""
    mb = ds.nbytes / 1e6
    print(f"{nome}: {mb:.1f} MB")
    if mb > limite_mb:
        print(f"  ATENÇÃO: mais de {limite_mb} MB! Reduza o período ou a área antes de carregar.")
    return mb


def mapa_base(ax, regiao, terra=True):
    """Prepara um eixo cartopy: extensão, costa, grade com rótulos.

    terra=True pinta os continentes de cinza POR CIMA dos campos (para campos oceânicos).
    """
    ax.set_extent([regiao["lon_o"], regiao["lon_l"], regiao["lat_s"], regiao["lat_n"]],
                  crs=ccrs.PlateCarree())
    if terra:
        ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=3)
    ax.coastlines("50m", linewidth=0.6, zorder=4)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3, edgecolor="0.4", zorder=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.5", linestyle="--", zorder=5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}
    return ax


def passo_vetores(n_pontos, n_setas=20):
    """Quantos pontos pular para desenhar ~n_setas vetores por eixo."""
    return max(1, n_pontos // n_setas)


def barra_cores(fig, mapeavel, rotulo):
    """Ajusta o espaço entre os painéis e põe UMA barra de cores horizontal embaixo da figura.

    Devolve a altura (fração da figura) da barra, útil para posicionar a seta de referência.
    """
    h = fig.get_figheight()
    y_barra = 0.8 / h
    fig.subplots_adjust(left=0.04, right=0.98, top=1 - 0.4 / h, bottom=y_barra + 1.0 / h,
                        wspace=0.18, hspace=0.3)
    cax = fig.add_axes([0.25, y_barra, 0.5, 0.15 / h])
    fig.colorbar(mapeavel, cax=cax, orientation="horizontal", label=rotulo)
    return y_barra


def rodape(fig, texto):
    """Escreve a fonte dos dados no rodapé da figura."""
    fig.text(0.01, 0.005, texto, fontsize=7, color="0.3", ha="left", va="bottom")


def ponto_oceano_mais_proximo(campo, lon, lat, nome_lon="longitude", nome_lat="latitude"):
    """Ponto de grade mais próximo de (lon, lat) que NÃO é terra (valor não-NaN).

    `campo` é um DataArray 2D (lat, lon) em que a terra está como NaN.
    Devolve (lon, lat) do ponto escolhido.
    """
    coslat = np.cos(np.deg2rad(lat))
    dist2 = ((campo[nome_lon] - lon) * coslat) ** 2 + (campo[nome_lat] - lat) ** 2
    dist2 = dist2.where(campo.notnull())
    idx = dist2.argmin(dim=[nome_lat, nome_lon])
    lon_ok = float(campo[nome_lon][idx[nome_lon]])
    lat_ok = float(campo[nome_lat][idx[nome_lat]])
    return lon_ok, lat_ok


def salvar_figura(fig, nome):
    """Salva a figura em PNG (200 dpi) na pasta de figuras."""
    caminho = PASTA_FIGURAS / nome
    fig.savefig(caminho, dpi=200, bbox_inches="tight")
    print("Figura salva:", caminho)

## 6. ERA5: abertura do armazenamento ARCO (Zarr na nuvem)

O **ARCO-ERA5** é uma cópia do ERA5 no Google Cloud em formato **Zarr**, *Analysis-Ready, Cloud-Optimized* (Abernathey et al., 2021). Em vez de baixar arquivos inteiros, abrimos o conjunto completo (1940 até hoje, horário, 0,25°) de forma **preguiçosa**: só os metadados são lidos agora. Os números só são transferidos quando chamamos `.load()`.

O armazenamento é dividido em pedaços (*chunks*) de **um instante × globo inteiro**, ótimo para **mapas**. Por isso:

- abrimos o armazenamento **uma única vez** (leva alguns segundos);
- recortamos **no espaço e no tempo antes** de carregar;
- nunca fazemos um laço hora a hora reabrindo o armazenamento.

O ERA5 "final" tem cerca de 3 meses de atraso. Os meses mais recentes são o **ERA5T** (preliminar), que pode ser corrigido depois.

In [ ]:
URL_ERA5 = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

era5 = xr.open_zarr(URL_ERA5, chunks=None, storage_options=dict(token="anon"))

print("ERA5 final:          ", era5.attrs["valid_time_start"], "a", era5.attrs["valid_time_stop"])
print("ERA5T (preliminar) até", era5.attrs["valid_time_stop_era5t"])
print("Número de variáveis: ", len(era5.data_vars))

### Nomes das variáveis

As variáveis do ARCO-ERA5 têm **nomes longos** (`mean_sea_level_pressure`, `10m_u_component_of_wind`...). Os nomes curtos do ECMWF (`msl`, `u10`) só aparecem nos atributos. A célula abaixo lista as variáveis com seus nomes curtos e unidades. Use-a para procurar outras variáveis que queira acrescentar.

In [ ]:
# >>> ALUNO: troque as palavras-chave para procurar outras variáveis
PALAVRAS = ["sea_level", "10m_", "sea_surface", "heat_flux", "radiation", "evaporation",
            "precipitation", "stress", "geopotential", "land_sea"]

for v in sorted(era5.data_vars):
    if any(p in v for p in PALAVRAS):
        atr = era5[v].attrs
        print(f"{v:58s} {atr.get('short_name', '?'):10s} [{atr.get('units', '?')}]")

### Recorte local horário (vento, TSM e tensão do vento)

No domínio local carregamos, **a cada hora**, no período que cobre o evento e a série temporal:

- vento a 10 m (`10m_u_component_of_wind`, `10m_v_component_of_wind`);
- TSM (`sea_surface_temperature`, em K). **Atenção:** no ERA5 a TSM **não é calculada pelo modelo**. Ela é uma condição de contorno vinda de análises de satélite e *in situ* (OSTIA, a partir de 2007), interpolada para 0,25°. Por isso tende a ser mais suave que a do GLORYS12;
- **tensão do vento turbulenta** na superfície: `mean_eastward_turbulent_surface_stress` e `mean_northward_turbulent_surface_stress` (N m⁻², média da hora anterior). Se o conjunto não tivesse a tensão, daria para estimá-la a partir do vento pela fórmula *bulk* (seção 9.1);
- máscara terra-mar.

São várias centenas de instantes × 5 variáveis, e **cada leitura traz um campo global**: esta célula leva alguns minutos.

In [ ]:
VARS_LOCAL = {
    "10m_u_component_of_wind": "u10",
    "10m_v_component_of_wind": "v10",
    "sea_surface_temperature": "sst",
    "mean_eastward_turbulent_surface_stress": "taux",
    "mean_northward_turbulent_surface_stress": "tauy",
    "land_sea_mask": "lsm",
}

e5_inicio = min(pd.Timestamp(INICIO), pd.Timestamp(SERIE_INICIO))
e5_fim = max(pd.Timestamp(FIM), pd.Timestamp(SERIE_FIM))

fim_era5 = pd.Timestamp(era5.attrs["valid_time_stop"]) + pd.Timedelta(hours=23)
if pd.Timestamp(e5_fim) > fim_era5:
    print("ATENÇÃO: parte do período está no ERA5T (preliminar), sujeito a correções.")

e5 = era5[list(VARS_LOCAL)].sel(
    time=slice(e5_inicio, e5_fim),
    latitude=slice(REGIAO["lat_n"], REGIAO["lat_s"]),
    longitude=slice(lon_para_360(REGIAO["lon_o"]), lon_para_360(REGIAO["lon_l"])),
).rename(VARS_LOCAL)
lsm = e5["lsm"].isel(time=0).drop_vars("time")
e5 = e5.drop_vars("lsm")
checar_tamanho(e5, "ERA5 local horário")

In [ ]:
e5 = lon_para_180(e5.load())
lsm = lon_para_180(lsm.load())
oceano = lsm < 0.5

e5["sst"] = e5["sst"] - 273.15
e5["sst"].attrs["units"] = "degC"

e5.to_netcdf(PASTA_DADOS / "t2_era5_local.nc")

## 7. CMEMS: GLORYS12 (superfície e seção vertical)

Usamos o serviço `arco-geo-series`, organizado para mapas (área grande, poucos instantes). Fazemos dois pedidos:

1. **Superfície, todos os dias do evento:** `thetao` (temperatura potencial, °C), `so` (salinidade, adimensional, na prática ≈ g/kg), `uo`, `vo` (m s⁻¹), no nível mais raso (~0,5 m).
2. **Perfil vertical (0 a `PROF_MAX`), só de `thetao` e `so` e só nos dois dias da seção** (antes e pico). No armazenamento ARCO, cada nível de profundidade de cada dia é um pedaço separado. Pedir as 4 variáveis em ~27 níveis por 19 dias leva mais de 20 minutos; pedir só o necessário leva segundos.

O aviso `WARNING ... depth` é esperado: o primeiro nível do modelo está em 0,49 m.

In [ ]:
ID_GLORYS = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

def abre_glorys(variaveis, inicio, fim, prof_max):
    return copernicusmarine.open_dataset(
        dataset_id=ID_GLORYS,
        variables=variaveis,
        minimum_longitude=REGIAO["lon_o"], maximum_longitude=REGIAO["lon_l"],
        minimum_latitude=REGIAO["lat_s"], maximum_latitude=REGIAO["lat_n"],
        minimum_depth=0, maximum_depth=prof_max,
        start_datetime=inicio, end_datetime=fim,
        service="arco-geo-series",
        coordinates_selection_method="outside",
    )

# 1) Superfície, todos os dias do evento
sup = abre_glorys(["thetao", "so", "uo", "vo"], INICIO, FIM, 1).isel(depth=0)
sup = sup.sel(time=slice(INICIO[:10], FIM[:10]))
checar_tamanho(sup, "GLORYS12 superfície")
sup = sup.load()
sup.to_netcdf(PASTA_DADOS / "t2_glorys_superficie.nc")
for v in sup.data_vars:
    print(f"{v:7s} [{sup[v].attrs.get('units')}]  {sup[v].attrs.get('standard_name')}")

**Dia do pico.** Se `DIA_PICO` é `None`, escolhemos o dia de TSM mínima do GLORYS12 no ponto de oceano mais próximo de `PONTO`, dentro da janela do evento.

In [ ]:
lon_g, lat_g = ponto_oceano_mais_proximo(sup["thetao"].isel(time=0), PONTO["lon"], PONTO["lat"])
tsm_ponto = sup["thetao"].sel(longitude=lon_g, latitude=lat_g)

if DIA_PICO is None:
    dia_pico = pd.Timestamp(tsm_ponto.idxmin("time").values).strftime("%Y-%m-%d")
else:
    dia_pico = DIA_PICO
dia_antes = DIA_ANTES
print(f"Dia 'antes': {dia_antes} | dia do pico: {dia_pico} "
      f"(TSM no ponto = {float(tsm_ponto.sel(time=dia_pico).squeeze()):.1f} °C)")

# 2) Perfis verticais de temperatura e salinidade, só nos dois dias da seção
perfis = {}
for d in [dia_antes, dia_pico]:
    g3d = abre_glorys(["thetao", "so"], f"{d}T00:00", f"{d}T00:00", PROF_MAX)
    checar_tamanho(g3d, f"GLORYS12 3D {d}")
    perfis[d] = g3d.isel(time=0).load()
    perfis[d].to_netcdf(PASTA_DADOS / f"t2_glorys_3d_{d}.nc")
print("Níveis:", perfis[dia_pico].sizes["depth"], "| prof. máxima:", float(perfis[dia_pico].depth.max()), "m")

## 8. Situação sinótica (ERA5)

### 8.1 Recorte sinótico (domínio grande, só os instantes dos mapas)

Para os mapas sinóticos bastam PNM e vento a 10 m nos instantes dos mapas: a cada `PASSO_MAPAS_H` horas, **mais o dia do pico às 12 UTC**, para que ele sempre apareça. Como cada chunk do ARCO-ERA5 é um campo global, o custo depende do **número de instantes × variáveis**, não do tamanho da área.

In [ ]:
DATAS_MAPAS = pd.date_range(INICIO, FIM, freq=f"{PASSO_MAPAS_H}h")
DATAS_MAPAS = DATAS_MAPAS.union([pd.Timestamp(dia_pico) + pd.Timedelta(hours=12)])
print(f"{len(DATAS_MAPAS)} mapas sinóticos, de {DATAS_MAPAS[0]} a {DATAS_MAPAS[-1]}")

VARS_SINOTICA = {
    "mean_sea_level_pressure": "msl",
    "10m_u_component_of_wind": "u10",
    "10m_v_component_of_wind": "v10",
}
e5s = era5[list(VARS_SINOTICA)].sel(
    time=DATAS_MAPAS,
    latitude=slice(REGIAO_SINOTICA["lat_n"], REGIAO_SINOTICA["lat_s"]),
    longitude=slice(lon_para_360(REGIAO_SINOTICA["lon_o"]), lon_para_360(REGIAO_SINOTICA["lon_l"])),
).rename(VARS_SINOTICA)
checar_tamanho(e5s, "ERA5 sinótico")
e5s = lon_para_180(e5s.load())
e5s["msl"] = e5s["msl"] / 100.0
e5s["msl"].attrs["units"] = "hPa"
e5s["vel10"] = np.hypot(e5s["u10"], e5s["v10"])
e5s.to_netcdf(PASTA_DADOS / "t2_era5_sinotico.nc")

### 8.2 Mapas

PNM em isolinhas (4 hPa) e vento a 10 m (cores = intensidade; vetores = direção). O retângulo marca o domínio local.

**O que observar:** a posição do ASAS e a sua borda oeste sobre o Sudeste, o vento de **NE** ao longo da costa do RJ e a chegada de **frentes frias** (cavados, giro do vento para S/SW) que interrompem a ressurgência.

In [ ]:
NCOL = 5   # >>> ALUNO
nlin = int(np.ceil(len(DATAS_MAPAS) / NCOL))
niveis_vel = np.arange(0, 22, 2)
niveis_pnm = np.arange(980, 1048, 4)
n = passo_vetores(e5s.sizes["longitude"])

fig, axs = plt.subplots(nlin, NCOL, figsize=(3.6 * NCOL, 3.6 * nlin),
                        subplot_kw=dict(projection=ccrs.PlateCarree()), squeeze=False)
for ax, t in zip(axs.flat, DATAS_MAPAS):
    c = e5s.sel(time=t)
    mapa_base(ax, REGIAO_SINOTICA, terra=False)
    cf = ax.contourf(c.longitude, c.latitude, c["vel10"], levels=niveis_vel,
                     cmap=cmocean.cm.speed, extend="max", transform=ccrs.PlateCarree())
    cs = ax.contour(c.longitude, c.latitude, c["msl"], levels=niveis_pnm,
                    colors="k", linewidths=0.7, transform=ccrs.PlateCarree())
    ax.clabel(cs, fmt="%d", fontsize=6)
    q = ax.quiver(c.longitude[::n], c.latitude[::n], c["u10"][::n, ::n], c["v10"][::n, ::n],
                  scale=250, width=0.0035, color="0.1", transform=ccrs.PlateCarree())
    ax.plot([REGIAO["lon_o"], REGIAO["lon_l"], REGIAO["lon_l"], REGIAO["lon_o"], REGIAO["lon_o"]],
            [REGIAO["lat_s"], REGIAO["lat_s"], REGIAO["lat_n"], REGIAO["lat_n"], REGIAO["lat_s"]],
            color="red", lw=1, transform=ccrs.PlateCarree(), zorder=6)
    marca = " (pico)" if t.strftime("%Y-%m-%d") == dia_pico else ""
    ax.set_title(f"PNM (hPa) e vento 10 m — {t:%Y-%m-%d %H} UTC{marca}", fontsize=8)
for ax in axs.flat[len(DATAS_MAPAS):]:
    ax.set_visible(False)

y = barra_cores(fig, cf, "Intensidade do vento a 10 m (m s$^{-1}$)")
ax.quiverkey(q, 0.85, y, 10, "10 m s$^{-1}$", labelpos="E", coordinates="figure")
rodape(fig, "Fonte: ERA5 (Hersbach et al., 2020), via ARCO-ERA5 (Google Cloud).")
salvar_figura(fig, f"t2_pnm_vento_{DATAS_MAPAS[0]:%Y-%m-%dT%H}_{DATAS_MAPAS[-1]:%Y-%m-%dT%H}.png")
plt.show()

## 9. Tensão do vento, transporte e bombeamento de Ekman

### 9.1 Fórmulas

**Tensão do vento.** Usamos a tensão turbulenta do ERA5 (τx, τy, em N m⁻²). Se ela não existisse, daria para estimá-la a partir do vento a 10 m pela fórmula *bulk*:

$$\vec{\tau} = \rho_{ar}\, C_D\, |\vec{U}|\, \vec{U}, \qquad \rho_{ar} = 1{,}22\ \mathrm{kg\,m^{-3}},\ C_D = 1{,}3\times10^{-3}$$

Um $C_D$ constante é uma **simplificação**: há formulações em que ele depende do vento (Large & Pond, 1981).

**Transporte de Ekman** por unidade de comprimento de costa (m² s⁻¹, ou seja, m³ s⁻¹ por metro de costa), com $f = 2\Omega\sin\varphi$ (negativo no Hemisfério Sul):

$$M_x = \frac{\tau_y}{\rho_{água}\, f}, \qquad M_y = -\frac{\tau_x}{\rho_{água}\, f}$$

Confira: com f < 0 e vento para leste (τx > 0), My > 0, isto é, transporte para o norte, **à esquerda** do vento.

**Índice de ressurgência** (IR; Bakun, 1973) = componente de $\vec{M}$ **para fora da costa**, na direção α + 90°:

$$IR = M_x \cos\alpha - M_y \sin\alpha$$

**IR > 0: transporte para fora da costa, favorável à ressurgência.**

**Tensão paralela à costa:** $\tau_\parallel = \tau_x \sin\alpha + \tau_y \cos\alpha$. A oeste de Cabo Frio (α ≈ 90°), τ∥ ≈ τx, e o vento de NE dá τ∥ < 0.

**Rotacional da tensão** em coordenadas esféricas (λ = longitude, φ = latitude, em radianos), calculado com diferenças centradas:

$$\mathrm{rot}(\vec\tau) = \frac{1}{R\cos\varphi}\left[\frac{\partial \tau_y}{\partial \lambda} - \frac{\partial (\tau_x\cos\varphi)}{\partial \varphi}\right]$$

**Bombeamento de Ekman:** velocidade vertical na base da camada de Ekman, $w_E = \frac{1}{\rho_{água}}\,\mathrm{rot}\!\left(\frac{\vec\tau}{f}\right)$. Como f varia só com a latitude ($\beta = \partial f/\partial y = 2\Omega\cos\varphi / R$):

$$w_E = \frac{1}{\rho_{água}}\left[\frac{\mathrm{rot}(\vec\tau)}{f} + \frac{\beta\,\tau_x}{f^2}\right]$$

O segundo termo (efeito β) é pequeno nesta região, mas fica no cálculo.

**w_E > 0: bombeamento para cima** (ressurgência por rotacional). No Hemisfério Sul (f < 0), isso corresponde a **rotacional negativo** da tensão. Mostramos w_E em m/dia.

In [ ]:
f = 2 * OMEGA * np.sin(np.deg2rad(e5.latitude))
alfa = np.deg2rad(ANGULO_COSTA_GRAUS)

# Transporte de Ekman (m2 s-1) e índice de ressurgência
e5["Mx"] = e5["tauy"] / (RHO_AGUA * f)
e5["My"] = -e5["taux"] / (RHO_AGUA * f)
e5["IR"] = e5["Mx"] * np.cos(alfa) - e5["My"] * np.sin(alfa)
e5["tau_par"] = e5["taux"] * np.sin(alfa) + e5["tauy"] * np.cos(alfa)

# Médias diárias da tensão (para os mapas)
tau_d = e5[["taux", "tauy"]].sel(time=slice(INICIO, FIM)).resample(time="1D").mean()

# Rotacional com métrica esférica. differentiate() usa diferenças centradas e
# deriva "por grau"; multiplicamos por 180/pi para derivar "por radiano".
coslat = np.cos(np.deg2rad(tau_d.latitude))
por_radiano = 180.0 / np.pi
dtauy_dlambda = tau_d["tauy"].differentiate("longitude") * por_radiano
dtauxcos_dphi = (tau_d["taux"] * coslat).differentiate("latitude") * por_radiano
rot = (dtauy_dlambda - dtauxcos_dphi) / (RAIO_TERRA * coslat)
tau_d["rot"] = rot.where(oceano)

# Bombeamento de Ekman, com o termo beta
beta = 2 * OMEGA * coslat / RAIO_TERRA
wE = (tau_d["rot"] / f + beta * tau_d["taux"] / f**2) / RHO_AGUA      # m/s
tau_d["wE"] = (wE * 86400).where(oceano)                              # m/dia
tau_d["wE"].attrs["units"] = "m/dia"

### 9.2 Mapas do bombeamento de Ekman

w_E (cores) e tensão do vento (vetores), na **média do evento** e no **dia do pico**. Com a grade de 0,25° do ERA5, o rotacional junto à costa é pouco resolvido, e as células mistas terra-mar foram mascaradas. Interprete o padrão de grande escala.

In [ ]:
campos = [("média do evento", tau_d.mean("time")),
          (f"{dia_pico} (pico)", tau_d.sel(time=dia_pico).squeeze())]
lim = float(np.nanpercentile(np.abs(tau_d["wE"]), 98))
lim = max(np.ceil(lim), 1.0)
niveis_w = np.linspace(-lim, lim, 21)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.8), subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, (rotulo, c) in zip(axs, campos):
    mapa_base(ax, REGIAO, terra=True)
    cf = ax.contourf(c.longitude, c.latitude, c["wE"], levels=niveis_w, cmap=cmocean.cm.balance,
                     extend="both", transform=ccrs.PlateCarree())
    q = ax.quiver(c.longitude, c.latitude, c["taux"].where(oceano), c["tauy"].where(oceano),
                  scale=4, width=0.004, color="k", transform=ccrs.PlateCarree())
    ax.plot(PONTO["lon"], PONTO["lat"], marker="*", ms=11, color="yellow", mec="k",
            transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(f"w$_E$ e tensão do vento — {rotulo}", fontsize=9)
y = barra_cores(fig, cf, "Bombeamento de Ekman w$_E$ (m dia$^{-1}$; > 0 = para cima)")
ax.quiverkey(q, 0.85, y, 0.1, "0,1 N m$^{-2}$", labelpos="E", coordinates="figure")
rodape(fig, "Fonte: ERA5 (Hersbach et al., 2020), via ARCO-ERA5. Tensão turbulenta média horária, médias diárias.")
salvar_figura(fig, f"t2_bombeamento_ekman_{INICIO[:10]}_{FIM[:10]}.png")
plt.show()

## 10. Resposta oceânica

### 10.1 TSM: ERA5 vs. GLORYS12

TSM no **dia do pico** e na **média do evento**, lado a lado e na mesma escala de cores. A linha tracejada é o transecto da seção vertical (seção 10.3).

**O que observar:** a língua de água fria junto a Cabo Frio e Arraial do Cabo; a diferença de **resolução** entre a TSM do ERA5 (0,25°, prescrita a partir de análises) e a do GLORYS12 (1/12°, calculada pelo modelo oceânico com assimilação de dados).

In [ ]:
# Coordenadas do transecto: da origem (PONTO) para fora da costa (azimute alfa + 90°)
azimute = np.deg2rad(ANGULO_COSTA_GRAUS + 90.0)
dist_km = np.arange(TRANSECTO_INICIO_KM, TRANSECTO_FIM_KM + 0.1, 4.0)
lat_t = PONTO["lat"] + dist_km * np.cos(azimute) / 111.2
lon_t = PONTO["lon"] + dist_km * np.sin(azimute) / (111.2 * np.cos(np.deg2rad(PONTO["lat"])))

sst_era5_pico = e5["sst"].sel(time=dia_pico).mean("time")
sst_era5_media = e5["sst"].sel(time=slice(INICIO, FIM)).mean("time")
sst_glorys_pico = sup["thetao"].sel(time=dia_pico).squeeze()
sst_glorys_media = sup["thetao"].mean("time")

paineis = [
    (sst_era5_pico, f"TSM ERA5 — {dia_pico} (média diária)"),
    (sst_glorys_pico, f"TSM GLORYS12 — {dia_pico} (média diária)"),
    (sst_era5_media, f"TSM ERA5 — média {INICIO[:10]} a {FIM[:10]}"),
    (sst_glorys_media, f"TSM GLORYS12 — média {INICIO[:10]} a {FIM[:10]}"),
]
vmin = np.floor(min(float(p.min()) for p, _ in paineis))
vmax = np.ceil(max(float(p.max()) for p, _ in paineis))
niveis_tsm = np.arange(vmin, vmax + 0.25, 0.5)

fig, axs = plt.subplots(2, 2, figsize=(10, 8.6), subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, (campo, titulo) in zip(axs.flat, paineis):
    mapa_base(ax, REGIAO, terra=True)
    cf = ax.contourf(campo.longitude, campo.latitude, campo, levels=niveis_tsm,
                     cmap=cmocean.cm.thermal, extend="both", transform=ccrs.PlateCarree())
    ax.plot(lon_t, lat_t, "w--", lw=1.2, transform=ccrs.PlateCarree(), zorder=6)
    ax.plot(PONTO["lon"], PONTO["lat"], marker="*", ms=11, color="yellow", mec="k",
            transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(titulo, fontsize=9)
barra_cores(fig, cf, "TSM (°C)")
rodape(fig, "Fontes: ERA5 (Hersbach et al., 2020), via ARCO-ERA5; GLORYS12 (Lellouche et al., 2021), CMEMS GLOBAL_MULTIYEAR_PHY_001_030.")
salvar_figura(fig, f"t2_tsm_era5_glorys_{dia_pico}.png")
plt.show()

### 10.2 Salinidade de superfície e correntes (GLORYS12)

Salinidade (cores) e correntes de superfície (vetores) **antes** e **no pico** do evento. A ACAS é mais salina que a Água Costeira, e a Água Tropical da Corrente do Brasil é ainda mais salina. Repare também no transporte para fora da costa.

In [ ]:
datas = [dia_antes, dia_pico]
campos = [sup.sel(time=d).squeeze() for d in datas]
vmin = np.floor(min(float(c["so"].quantile(0.01)) for c in campos) * 4) / 4
vmax = np.ceil(max(float(c["so"].quantile(0.99)) for c in campos) * 4) / 4
niveis_sss = np.arange(vmin, vmax + 0.05, 0.1)
n = passo_vetores(sup.sizes["longitude"])

fig, axs = plt.subplots(1, 2, figsize=(10, 4.8), subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, c, d in zip(axs, campos, datas):
    mapa_base(ax, REGIAO, terra=True)
    cf = ax.contourf(c.longitude, c.latitude, c["so"], levels=niveis_sss, cmap=cmocean.cm.haline,
                     extend="both", transform=ccrs.PlateCarree())
    q = ax.quiver(c.longitude[::n], c.latitude[::n], c["uo"][::n, ::n], c["vo"][::n, ::n],
                  scale=8, width=0.004, color="k", transform=ccrs.PlateCarree())
    ax.plot(PONTO["lon"], PONTO["lat"], marker="*", ms=11, color="yellow", mec="k",
            transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(f"Salinidade e correntes de superfície — {d}", fontsize=9)
y = barra_cores(fig, cf, "Salinidade de superfície (GLORYS12, adimensional ≈ g kg$^{-1}$)")
ax.quiverkey(q, 0.85, y, 0.5, "0,5 m s$^{-1}$", labelpos="E", coordinates="figure")
rodape(fig, "Fonte: GLORYS12 (Lellouche et al., 2021), CMEMS GLOBAL_MULTIYEAR_PHY_001_030, via Copernicus Marine Toolbox (ARCO).")
salvar_figura(fig, f"t2_sss_correntes_{dia_antes}_{dia_pico}.png")
plt.show()

### 10.3 Seções verticais de temperatura e salinidade (GLORYS12)

Temperatura e salinidade × profundidade ao longo do transecto perpendicular à costa (para o sul, a partir do ponto), **antes** e **no pico**. As isotermas de **18 °C** e **20 °C** estão destacadas como **referência visual** da ACAS. Esse limiar é **indicativo**, não uma definição rígida: a ACAS costuma ser definida por faixas de temperatura e salinidade (diagrama T-S), e os limiares variam entre autores. Nas seções de salinidade, as mesmas isotermas aparecem sobrepostas, para comparar a posição da água fria com a da água mais salina. Cinza = fundo (fora da grade do modelo).

**O que observar:** as isotermas sobem em direção à costa ("inclinam para cima") quando há ressurgência. Compare com o cruzeiro de 20–21/01/2016 (Ramos e Silva et al., 2023), que encontrou 13,9–14,7 °C junto ao fundo.

In [ ]:
pontos_lon = xr.DataArray(lon_t, dims="dist", coords={"dist": dist_km})
pontos_lat = xr.DataArray(lat_t, dims="dist", coords={"dist": dist_km})

secoes = {d: perfis[d].interp(longitude=pontos_lon, latitude=pontos_lat) for d in [dia_antes, dia_pico]}
niveis_t = np.arange(12, 29, 0.5)
smin = np.floor(min(float(sc["so"].min()) for sc in secoes.values()) * 10) / 10
smax = np.ceil(max(float(sc["so"].max()) for sc in secoes.values()) * 10) / 10
niveis_s = np.arange(smin, smax + 0.01, 0.05)

fig, axs = plt.subplots(2, 2, figsize=(11, 8.4), sharex=True, sharey=True)
for j, d in enumerate([dia_antes, dia_pico]):
    secao = secoes[d]
    for i, (v, niveis, cmap, rotulo) in enumerate([("thetao", niveis_t, cmocean.cm.thermal, "Temperatura"),
                                                    ("so", niveis_s, cmocean.cm.haline, "Salinidade")]):
        ax = axs[i, j]
        ax.set_facecolor("0.6")      # fundo / terra
        cf = ax.contourf(secao.dist, secao.depth, secao[v], levels=niveis, cmap=cmap, extend="both")
        cs = ax.contour(secao.dist, secao.depth, secao["thetao"], levels=[18, 20], colors=["k", "w"],
                        linewidths=1.8)
        ax.clabel(cs, fmt="%d °C", fontsize=8)
        ax.set_ylim(PROF_MAX, 0)
        ax.set_title(f"{rotulo} (GLORYS12) — {d}", fontsize=9)
        if j == 1:
            fig.colorbar(cf, ax=axs[i, :], shrink=0.9,
                         label="Temperatura potencial (°C)" if v == "thetao" else "Salinidade (≈ g kg$^{-1}$)")
for ax in axs[:, 0]:
    ax.set_ylabel("Profundidade (m)")
for ax in axs[1, :]:
    ax.set_xlabel("Distância a partir do ponto, para fora da costa (km)")
rodape(fig, "Fonte: GLORYS12 (Lellouche et al., 2021), CMEMS GLOBAL_MULTIYEAR_PHY_001_030. Isotermas de 18 e 20 °C: referência visual da ACAS.")
salvar_figura(fig, f"t2_secao_temperatura_salinidade_{dia_antes}_{dia_pico}.png")
plt.show()

## 11. Série temporal no ponto

- **GLORYS12:** chamada separada com `service="arco-time-series"` (chunks longos no tempo) numa caixa pequena em volta do ponto, no período `SERIE_INICIO`–`SERIE_FIM`.
- **ERA5:** extraímos o ponto de oceano mais próximo do recorte local horário já carregado. O ARCO-ERA5 não tem versão organizada para séries temporais, então **evite pedir anos de dados horários**.

Painéis: (a) tensão paralela à costa; (b) índice de ressurgência (transporte de Ekman para fora da costa); (c) TSM do GLORYS12 (diária) e do ERA5 (horária). A faixa cinza marca a janela do evento e a linha tracejada, o cruzeiro de 20–21/01/2016.

In [ ]:
CAIXA = 0.3   # meia-largura (graus) da caixa em volta do ponto

st_glorys = copernicusmarine.open_dataset(
    dataset_id=ID_GLORYS, variables=["thetao"],
    minimum_longitude=PONTO["lon"] - CAIXA, maximum_longitude=PONTO["lon"] + CAIXA,
    minimum_latitude=PONTO["lat"] - CAIXA, maximum_latitude=PONTO["lat"] + CAIXA,
    minimum_depth=0, maximum_depth=1,
    start_datetime=SERIE_INICIO, end_datetime=SERIE_FIM,
    service="arco-time-series",           # chunks otimizados para séries temporais
    coordinates_selection_method="outside",
).isel(depth=0).load()

lon_gs, lat_gs = ponto_oceano_mais_proximo(st_glorys["thetao"].isel(time=0), PONTO["lon"], PONTO["lat"])
# ERA5: ponto mais próximo com TSM válida (perto da costa, o ERA5 deixa a TSM em branco)
lon_e, lat_e = ponto_oceano_mais_proximo(e5["sst"].isel(time=0), PONTO["lon"], PONTO["lat"])
pt_glorys = st_glorys["thetao"].sel(longitude=lon_gs, latitude=lat_gs)
pt_era5 = e5.sel(longitude=lon_e, latitude=lat_e).sel(time=slice(SERIE_INICIO, SERIE_FIM))

print(f"Ponto pedido:   {PONTO['lon']:.2f}, {PONTO['lat']:.2f}")
print(f"Ponto GLORYS12: {lon_gs:.2f}, {lat_gs:.2f}")
print(f"Ponto ERA5:     {lon_e:.2f}, {lat_e:.2f}")

In [ ]:
CRUZEIRO = ("2016-01-20", "2016-01-21T23:59")   # >>> ALUNO: datas de observações in situ (ou None)

fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

ax = axs[0]
ax.plot(pt_era5.time, pt_era5["tau_par"], color="0.6", lw=0.8, label="horária")
ax.plot(pt_era5.time, pt_era5["tau_par"].rolling(time=24, center=True).mean(), color="k",
        label="média móvel de 24 h")
ax.axhline(0, color="0.5", lw=0.6)
ax.set_ylabel(r"$\tau_\parallel$ (N m$^{-2}$)")
ax.legend(fontsize=7, loc="upper left")
ax.set_title(f"(a) Tensão paralela à costa (α = {ANGULO_COSTA_GRAUS:.0f}°; < 0 = vento de NE) — ERA5",
             loc="left", fontsize=9)

ax = axs[1]
ir24 = pt_era5["IR"].rolling(time=24, center=True).mean()
ax.plot(pt_era5.time, pt_era5["IR"], color="0.6", lw=0.8)
ax.fill_between(pt_era5.time.values, 0, ir24, where=ir24 > 0, color="tab:blue", alpha=0.4,
                label="favorável à ressurgência")
ax.fill_between(pt_era5.time.values, 0, ir24, where=ir24 < 0, color="tab:red", alpha=0.4,
                label="desfavorável (para a costa)")
ax.plot(pt_era5.time, ir24, color="k")
ax.axhline(0, color="0.5", lw=0.6)
ax.set_ylabel("IR (m$^2$ s$^{-1}$)")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("(b) Índice de ressurgência: transporte de Ekman para fora da costa — ERA5",
             loc="left", fontsize=9)

ax = axs[2]
ax.plot(pt_era5.time, pt_era5["sst"], color="tab:orange", label="ERA5 (horária, prescrita)")
ax.plot(pt_glorys.time + pd.Timedelta(hours=12), pt_glorys, "o-", color="tab:blue", ms=3,
        label="GLORYS12 (média diária)")
ax.set_ylabel("TSM (°C)")
ax.legend(fontsize=7, loc="lower left")
ax.set_title("(c) Temperatura da superfície do mar", loc="left", fontsize=9)

for ax in axs:
    ax.axvspan(pd.Timestamp(INICIO), pd.Timestamp(FIM), color="0.92", zorder=0)
    if CRUZEIRO:
        for d in CRUZEIRO:
            ax.axvline(pd.Timestamp(d), color="tab:purple", ls="--", lw=0.8)
    ax.grid(lw=0.3)
axs[-1].set_xlabel("Tempo (UTC)")
axs[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axs[-1].xaxis.get_major_locator()))
fig.suptitle(f"{PONTO['nome']} ({PONTO['lon']:.2f}°, {PONTO['lat']:.2f}°)", y=0.995)
fig.tight_layout()
rodape(fig, "Fontes: ERA5 (Hersbach et al., 2020); GLORYS12 (Lellouche et al., 2021).")
salvar_figura(fig, f"t2_serie_ponto_{SERIE_INICIO[:10]}_{SERIE_FIM[:10]}.png")
plt.show()

### 11.1 Defasagem entre vento e resfriamento

O oceano não responde instantaneamente: a TSM leva alguns dias para cair depois que o vento favorável se instala. **Estime essa defasagem** primeiro a olho, na figura acima. Depois compare com a estimativa simples abaixo, que calcula a correlação entre o IR médio diário e a TSM diária do GLORYS12 com o vento "adiantado" de 0 a 10 dias. Espera-se **correlação negativa** (mais transporte para fora da costa, TSM mais fria), mais forte em alguma defasagem positiva.

Isto é uma estimativa grosseira, com poucos dias de dados e sem teste de significância. Discuta o resultado em vez de tomá-lo como definitivo.

In [ ]:
ir_diario = pt_era5["IR"].resample(time="1D").mean().to_series()
tsm_diaria = pt_glorys.to_series()
tsm_diaria.index = tsm_diaria.index.normalize()

defasagens = range(0, 11)   # >>> ALUNO: defasagens testadas (dias)
correl = [tsm_diaria.corr(ir_diario.shift(k)) for k in defasagens]

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(defasagens), correl, color="tab:blue")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("Defasagem: vento antes da TSM (dias)")
ax.set_ylabel("Correlação IR × TSM")
ax.set_title("Correlação defasada (estimativa simples)", fontsize=9)
salvar_figura(fig, "t2_defasagem_ir_tsm.png")
plt.show()
k_min = list(defasagens)[int(np.nanargmin(correl))]
print(f"Correlação mais negativa: {np.nanmin(correl):.2f} com defasagem de {k_min} dia(s)")

## 12. Para ir além

- **Nutrientes e clorofila (modelo):** o GLORYS12 **não tem biogeoquímica**. O CMEMS oferece o produto biogeoquímico multianual global **GLOBAL_MULTIYEAR_BGC_001_029** (`cmems_mod_glo_bgc_my_0.25deg_P1D-m`, diário), com nitrato (`no3`), fosfato (`po4`), silicato (`si`), oxigênio (`o2`), clorofila (`chl`) e produção primária (`nppv`). Ele pode ser aberto da mesma forma que o GLORYS, com `copernicusmarine.open_dataset`. **Atenção:** a resolução é de **0,25°**, bem mais grosseira que a do GLORYS12 (1/12°). A ressurgência costeira de Cabo Frio, com poucas dezenas de km de largura, fica mal resolvida.
- **Clorofila por satélite:** produtos de cor do oceano do CMEMS (OCEANCOLOUR) ou da NASA (MODIS-Aqua) mostram a resposta biológica.
- **TSM por satélite de alta resolução:** produtos L4 (por exemplo, OSTIA ou MUR), para comparar com o ERA5 e o GLORYS12.
- **Observações in situ:** compare a seção vertical com os perfis do cruzeiro de 20–21/01/2016 (Ramos e Silva et al., 2023) e com boias do PNBoia ou do projeto SIMCosta, quando houver.
- **Vento por satélite (ASCAT)** para validar o vento do ERA5 junto à costa.
- **Outros episódios:** troque as datas para um evento de ressurgência forte (primavera/verão) e compare o IR, o w_E e a resposta da TSM.

## Opcional: copiar dados e figuras para o Google Drive

A pasta `/content` some ao fim da sessão do Colab. Para guardar os resultados, mude `COPIAR_PARA_DRIVE` para `True` e rode a célula. O Colab vai pedir autorização para acessar o seu Drive.

In [ ]:
COPIAR_PARA_DRIVE = False   # >>> ALUNO: True para copiar
PASTA_NO_DRIVE = "ACA0430_trabalho1"   # >>> ALUNO: nome da pasta dentro de "Meu Drive"

if EM_COLAB and COPIAR_PARA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive") / PASTA_NO_DRIVE
    shutil.copytree(PASTA_DADOS, destino / "dados", dirs_exist_ok=True)
    shutil.copytree(PASTA_FIGURAS, destino / "figuras", dirs_exist_ok=True)
    print("Copiado para", destino)

## 13. Referências

**Dados e infraestrutura**
- Hersbach, H., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146, 1999–2049. https://doi.org/10.1002/qj.3803
- Lellouche, J.-M., et al. (2021). The Copernicus Global 1/12° Oceanic and Sea Ice GLORYS12 Reanalysis. *Frontiers in Earth Science*, 9, 698876. https://doi.org/10.3389/feart.2021.698876
- Law-Chune, S., Aouf, L., Dalphinet, A., Levier, B., Drillet, Y., & Drevillon, M. (2021). WAVERYS: a CMEMS global wave reanalysis during the altimetry period. *Ocean Dynamics*, 71, 357–378. https://doi.org/10.1007/s10236-020-01433-w
- Abernathey, R. P., et al. (2021). Cloud-native repositories for big scientific data. *Computing in Science & Engineering*, 23(2), 26–35. https://doi.org/10.1109/MCSE.2021.3059437
- Documentação técnica (não revisada por pares, citar como fonte dos dados): repositório ARCO-ERA5 (https://github.com/google-research/arco-era5); Manual do Usuário CMEMS-GLO-PUM-001-030; página do produto GLOBAL_MULTIYEAR_WAV_001_032 (DOI 10.48670/moi-00022); documentação do Copernicus Marine Toolbox (https://toolbox-docs.marine.copernicus.eu).

**Tema**
- Bakun, A. (1973). *Coastal upwelling indices, west coast of North America, 1946–71*. NOAA Technical Report NMFS SSRF-671. U.S. Department of Commerce, 103 p.
- Castelao, R. M., & Barth, J. A. (2006). Upwelling around Cabo Frio, Brazil: The importance of wind stress curl. *Geophysical Research Letters*, 33, L03602. https://doi.org/10.1029/2005GL025182
- Large, W. G., & Pond, S. (1981). Open ocean momentum flux measurements in moderate to strong winds. *Journal of Physical Oceanography*, 11, 324–336. https://doi.org/10.1175/1520-0485(1981)011<0324:OOMFMI>2.0.CO;2
- Ramos e Silva, et al. (2023). [completar título]. *Scientific Reports*, [volume, artigo]. [completar DOI]